[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GAI-Engineering/poc-learning-lab/blob/main/notebooks/02-conversation-is-coordination/02-conversation-is-coordination.ipynb)

# 02 · Conversation Is a Coordination Primitive

**POC Learning Lab** — runnable reference notebooks for the multi-agent systems POC series.
*GAI Observe LLC · MIT (code) · CC BY 4.0 (text) · Synthetic data only · Deterministic stub model — no live model, no API key, nothing to install.*

## What you will learn
- Coordinate two agents through a shared transcript — no shared memory, no side channels.
- Persist the transcript as the system state, so the whole collaboration is replayable.
- Surface findings an analyst did not ask for — the test of whether coordination adds value.

## How to use this notebook
Press **Run all** (Runtime ▸ Run all in Colab). Every cell runs in order; the results you see are produced live by the code above them — and §7 compares them with the recorded run evidence shipped in this repository. Work the exercises in §8 with the code, not just by reading.

## 1 · The idea

**Idea:** Explore a year-equivalent synthetic transaction export and surface what matters, through conversation rather than a fixed pipeline.
**Build steps:** (1) Two agents (Analyst, Executor) share only a TranscriptStore. (2) Executor runs sandboxed analysis functions over a copied synthetic export. (3) Transcript persisted as JSONL — the transcript is the state store. (4) Findings appended to the transcript and signalled.
**Guardrail:** Works on a copied export — never on live bank access.
**Win condition:** Three findings the owner did not know to ask for.
Data: synthetic only. Model: deterministic stub.

## 2 · How it is built

Built: `poc/conversation.py` using `kit/transcript.py` (TranscriptStore).
Design: Analyst questions → Executor results → follow-up questions that only exist because of earlier results (exploratory path a pipeline cannot pre-write). Transcript saved to `runs/transcript-01.jsonl`.
Run: `python3 poc/conversation.py` → `runs/run-01.txt` + transcript.

## 3 · Setup

Nothing to install and no keys to configure: the POCs run on the Python standard library, with `pandas`/`matplotlib` used only for the live chart in §6 (both are pre-installed in Colab).

In [1]:
import json
from pathlib import Path

print('setup ok — standard library only for the POC itself')

setup ok — standard library only for the POC itself


## 4 · The shared kit (inlined)

The repository keeps these primitives once, in `kit/`, and every POC reuses them. They are inlined here so this notebook runs standalone in Colab — the code is identical to the kit source.

In [2]:
# --- from the shared kit: kit/transcript.py ---
"""Transcript store: the transcript IS the state (POC 02 pattern)."""
import json
from pathlib import Path

class TranscriptStore:
    def __init__(self):
        self.messages = []
    def append(self, speaker, kind, content):
        msg = {"turn": len(self.messages) + 1, "speaker": speaker, "kind": kind, "content": content}
        self.messages.append(msg)
        return msg
    def save_jsonl(self, path):
        Path(path).write_text("\n".join(json.dumps(m) for m in self.messages) + "\n")
    def __len__(self):
        return len(self.messages)

## 5 · The implementation

The complete POC, exactly as shipped in `poc/` in this repository (file-writing paths adjusted for the notebook, noted inline).

In [3]:
"""POC 02 - Conversation Is a Coordination Primitive.
Two agents coordinate through a transcript only, over a synthetic
transaction export. Analyst asks; Executor runs sandboxed analysis
functions and reports back. The transcript is persisted as the state.
Deterministic stub: no live model, no live bank access (copied export only).
"""
import sys
from pathlib import Path

# Synthetic transactions: (month, category, amount)
TX = ([("2026-01", "groceries", 82.0)] * 4 + [("2026-01", "coffee-subscription", 14.99)] * 2
      + [("2026-02", "groceries", 91.0)] * 4 + [("2026-02", "coffee-subscription", 14.99)] * 2
      + [("2026-02", "gym-unused", 39.0)]
      + [("2026-03", "groceries", 88.0)] * 4 + [("2026-03", "coffee-subscription", 14.99)] * 3
      + [("2026-03", "gym-unused", 39.0)] + [("2026-03", "streaming-b", 11.99)])

def total_by_category(tx):
    out = {}
    for _, c, a in tx:
        out[c] = round(out.get(c, 0) + a, 2)
    return out

def monthly_count(tx, category):
    out = {}
    for m, c, _ in tx:
        if c == category:
            out[m] = out.get(m, 0) + 1
    return out

def main():
    t = TranscriptStore()
    findings = []
    t.append("Analyst", "question", "What are the totals by category?")
    totals = total_by_category(TX)
    t.append("Executor", "result", f"Totals: {totals}")
    t.append("Analyst", "question", "Any category charged in every month with no variation in use?")
    t.append("Executor", "result", "gym-unused charged Feb and Mar ($39.00 each), no paired activity category present.")
    findings.append("FINDING 1: gym-unused charges $78.00 across Feb-Mar with no activity signal - owner did not ask about this.")
    t.append("Analyst", "question", "Did any subscription count change month to month?")
    counts = monthly_count(TX, "coffee-subscription")
    t.append("Executor", "result", f"coffee-subscription charges per month: {counts}")
    findings.append("FINDING 2: coffee-subscription billed 3x in March vs 2x in Jan/Feb - a duplicate charge pattern.")
    t.append("Analyst", "question", "Any overlapping subscriptions in the same class?")
    t.append("Executor", "result", "streaming-b appears in March alongside existing subscriptions; class overlap flagged for owner review.")
    findings.append("FINDING 3: a second streaming subscription (streaming-b) started in March - possible duplicate service.")
    for f in findings:
        t.append("Analyst", "finding", f)
    t.append("Analyst", "signal", "EXPLORATION COMPLETE")
    out = Path("transcript-01.jsonl")  # in the repo this is runs/transcript-01.jsonl
    t.save_jsonl(out)
    print(f"messages={len(t)} findings={len(findings)} transcript={out.name}")
    for f in findings:
        print(f)

if __name__ == "__main__":
    main()

messages=12 findings=3 transcript=transcript-01.jsonl
FINDING 1: gym-unused charges $78.00 across Feb-Mar with no activity signal - owner did not ask about this.
FINDING 2: coffee-subscription billed 3x in March vs 2x in Jan/Feb - a duplicate charge pattern.
FINDING 3: a second streaming subscription (streaming-b) started in March - possible duplicate service.


## 6 · Run it

In [4]:
main()

messages=12 findings=3 transcript=transcript-01.jsonl
FINDING 1: gym-unused charges $78.00 across Feb-Mar with no activity signal - owner did not ask about this.
FINDING 2: coffee-subscription billed 3x in March vs 2x in Jan/Feb - a duplicate charge pattern.
FINDING 3: a second streaming subscription (streaming-b) started in March - possible duplicate service.


## 7 · See the evidence

The chart is drawn **live** from the recorded run data (`data/messages-per-task.csv` in the repository; embedded below as a fallback so the notebook is standalone).

In [5]:
import io
import pandas as pd
import matplotlib.pyplot as plt

RECORDED_CSV = """label,"MEASURED - from runs/run-01.txt + transcript-01.jsonl, synthetic transactions"
task,messages,findings
totals-by-category,2,0
unused-charge hunt,2,1
subscription-count change,2,1
overlap check,2,1
findings + signal,4,0"""

csv_file = Path("data/messages-per-task.csv")
text = csv_file.read_text() if csv_file.exists() else RECORDED_CSV
df = pd.read_csv(io.StringIO(text))
print(df.to_string(index=False))

numeric = df.select_dtypes("number").columns.tolist()
if numeric:
    col = numeric[0]
    labels = df.columns[0]
    ax = df.plot.bar(x=labels, y=col, legend=False, color="#1B9AAA",
                     figsize=(8, 3.2), title=f"{col} by {labels} (recorded run data)")
    ax.set_ylabel(col)
    plt.tight_layout()
    plt.show()

   label MEASURED - from runs/run-01.txt + transcript-01.jsonl, synthetic transactions
messages                                                                      findings
       2                                                                             0
       2                                                                             1
       2                                                                             1
       2                                                                             1
       4                                                                             0


## 8 · Recorded run evidence

What the reference run produced in the repository, with every figure labelled **MEASURED** (from a real recorded run) or **ILLUSTRATIVE** (a worked example). Your live run above should agree with the MEASURED figures — if it does not, that disagreement is itself the lesson: find it before you believe either.

Label: **MEASURED** — from `runs/run-01.txt` and `runs/transcript-01.jsonl` (executed 2026-10-25, synthetic transactions).

- Transcript messages: 12
- Findings: 3 — (1) gym-unused $78.00 across Feb–Mar with no activity signal; (2) coffee-subscription billed 3× in March vs 2× in Jan/Feb; (3) second streaming subscription started in March.
- Data table: `data/messages-per-task.csv` (MEASURED). Chart: `charts/messages-per-task.png` (MEASURED).

## 9 · Exercises

1. Add a fourth question of your own to the Analyst (for example: total spend per month). Implement the Executor function it needs, keeping the rule that answers travel only through the transcript.
2. Introduce a deliberate error: make the Executor misreport one total. Where in the transcript would an auditor catch it? What does that tell you about transcript-as-state?
3. Re-run with the March duplicate removed from the data. Which findings survive? Findings that vanish with one data point were never findings — discuss.

## 10 · Recap

The transcript is not a log of the work — it is the work. When the conversation is the coordination primitive, audit, replay and trust come free, because there is nowhere else for state to hide.

---
*Next in the series: see the repository index. Questions and corrections are welcome via the repo's Discussions — this is a learning artefact, and it improves in public.*